# SDK 0.3.1 four-L4 acceptance
Fresh Kaggle CPython 3.12.13, four NVIDIA L4, Internet disabled. Attach this small handoff, corrected split-v2 runtime Inputs, native P31 wheel or P31 final-v2 Notebook Output, and Gemma 4 QAT model Input. Do not attach historical SDK acceptance dataset if it creates duplicate wheel names. Download the launcher log and evidence ZIP. The SDK 0.3.1 release is already public. This run is fresh integration acceptance; do not treat the published release as proof of this run.


In [ ]:
from pathlib import Path
import os, subprocess, sys
print("notebook_python", sys.version.split()[0], sys.executable)
candidates = ["/usr/bin/python3", "/usr/bin/python3.12", "/usr/local/bin/python3.12", "/opt/conda/bin/python3.12"]
versions = {}
for executable in candidates:
    if Path(executable).is_file():
        result = subprocess.run([executable, "-c", "import sys; print(sys.version.split()[0])"], capture_output=True, text=True, timeout=10)
        versions[executable] = result.stdout.strip() if result.returncode == 0 else result.stderr.strip()
print("candidate_pythons", versions)
python312 = next((path for path, version in versions.items() if version.startswith("3.12.")), None)
assert python312, "No CPython 3.12 interpreter on this Kaggle image; native cp312 acceptance cannot run"
root = Path("/kaggle/input/datasets/waqasm86")
runners = sorted(root.rglob("production_acceptance_new_sdk_v031.py"))
assert len(runners) == 1, f"Expected one new SDK runner, found {len(runners)}"
runner = runners[0]
sdk = runner.with_name("kaggle_vllm-0.3.1-py3-none-any.whl")
assert sdk.is_file(), f"new SDK wheel missing next to runner: {sdk}"
pip_wheel = runner.with_name("pip-25.2-py3-none-any.whl")
assert pip_wheel.is_file(), "offline Python 3.12 pip bootstrap wheel missing"
env = dict(os.environ)
env["PYTHONPATH"] = os.pathsep.join(filter(None, (str(pip_wheel), env.get("PYTHONPATH", ""))))
probe = subprocess.run([python312, "-m", "pip", "--version"], env=env, capture_output=True, text=True, timeout=20)
assert probe.returncode == 0 and "pip 25.2" in probe.stdout, probe.stderr
print("offline_pip", probe.stdout.strip())
log = Path("/kaggle/working/new-sdk-acceptance-launcher.log")
with log.open("w", encoding="utf-8") as stream:
    completed = subprocess.run([python312, str(runner), "--sdk-wheel", str(sdk)], stdout=stream, stderr=subprocess.STDOUT, env=env, timeout=3600)
print("runner_exit", completed.returncode)
print("launcher_log", log)
print("log_tail", "\n".join(log.read_text(errors="replace").splitlines()[-30:]))
assert completed.returncode == 0, "Fresh acceptance failed; download evidence and launcher log"
